# Download Snetinel-2 imagery for prediction

In [ ]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

# Value to edit

In [ ]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]


In [ ]:
predict_max_cloud_covers = [1] # percentage
predict_tide_deltas = {1: {"hrs": 0, "mins": 30}} # {1: {"hrs": 0, "mins": 30},  2: {"hrs": 0, "mins": 45},  3: {"hrs": 1, "mins": 0}, 4: {"hrs": 1, "mins": 15}}
filtered=True

In [ ]:
prediction_sites = all_training_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()


# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

In [ ]:
all_baselines = sentinel2.load_band_quality_baseline()

for tide_delta in predict_tide_deltas.values():
    for max_cloud_cover in predict_max_cloud_covers:
        print(f"Max cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        predictions_path, satellite_path = utils.get_prediction_path(
            sample_method="sampling_2", method_2_threshold=0.5,
            model_max_cloud_cover=max_cloud_cover,
            predict_max_cloud_cover=max_cloud_cover,
            predict_low_tide_delta_hrs=tide_delta["hrs"],
            predict_low_tide_delta_mins=tide_delta["mins"],
            use_threshold=False, satellite_only=True,
            filtered=filtered)
        
        for prediction_site in prediction_sites:

            site_baseline = all_baselines.loc[all_baselines["site"] == prediction_site]
            
            for year in years:
                print(f"{prediction_site}: year {year}")
            
                satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
                site_polygon_file = utils.get_site_polygon_path(prediction_site)

                if (satellite_path / f"{prediction_site}_{years[-1]}_sentinel-2.nc").exists():
                    print(f"\tAlready run for {prediction_site} as {prediction_site}_{years[-1]}_sentinel-2.nc exists. "
                          "Delete if you want redownloaded.")
                    break
            
                if not satellite_file.exists():
                    print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
                    site_polygon = geopandas.read_file(site_polygon_file)
                    for i in range(3):
                        try:
                            satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                                geometry=site_polygon,
                                max_cloud_cover=max_cloud_cover,
                                year=year,
                                low_tide_delta_hrs=tide_delta["hrs"],
                                low_tide_delta_mins=tide_delta["mins"],
                                anomaly_filter_baseline=site_baseline,
                                n_mad=3,
                            )
                            break
                        except Exception:
                            print("Caught exception downloading images and try again.")
                    if len(satellite_data['time']) == 0:
                        print("Warning: No satellite data without cloud. Ignore")
                    else:
                        print("\t\tSave")
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                        satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                        utils.write_netcdf_conventions_in_place(satellite_data)
                        utils.save_netcdf(satellite_data, satellite_file)
                    utils.rgb_from_satellite(satellite_data_path=satellite_file)